In [1]:
import os
import shutil
from pyspark import SparkConf, SparkContext

# Initialize Spark Context
conf = SparkConf().setAppName("EmployeeDataProcessing").setMaster("local[*]")
sc = SparkContext.getOrCreate(conf=conf)

# Read CSV and skip the header row
raw_rdd = sc.textFile("employees.csv")
header = raw_rdd.first()
data_rdd = raw_rdd.filter(lambda line: line != header)

# Parse lines into structured tuples: (id, name, department, salary)
def parse_line(line):
    parts = line.split(',')
    return (int(parts[0]), parts[1], parts[2], float(parts[3]))

emp_rdd = data_rdd.map(parse_line).cache()
print("Spark Context initialized and data loaded successfully.")


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/20 12:02:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
[Stage 0:>                                                          (0 + 1) / 1]

Spark Context initialized and data loaded successfully.


In [2]:
# Sort all employees by salary in descending order
sorted_emp = emp_rdd.sortBy(lambda emp: emp[3], ascending=False).collect()

print("="*50)
print("EMPLOYEES SORTED BY SALARY (DESCENDING):")
print("="*50)
for emp in sorted_emp:
    print(f"ID: {emp[0]} | Name: {emp[1]} | Dept: {emp[2]} | Salary: {emp[3]}")


EMPLOYEES SORTED BY SALARY (DESCENDING):
ID: 4 | Name: Priya | Dept: Finance | Salary: 70000.0
ID: 3 | Name: Neha | Dept: IT | Salary: 65000.0
ID: 7 | Name: Rohit | Dept: Finance | Salary: 60000.0
ID: 1 | Name: Amit | Dept: IT | Salary: 55000.0
ID: 5 | Name: Karan | Dept: IT | Salary: 50000.0
ID: 6 | Name: Simran | Dept: HR | Salary: 45000.0
ID: 2 | Name: Rahul | Dept: HR | Salary: 40000.0


In [3]:
# Calculate total salary paid in each department
dept_salary_rdd = emp_rdd.map(lambda emp: (emp[2], emp[3]))
dept_totals = dept_salary_rdd.reduceByKey(lambda a, b: a + b).collect()

print("="*50)
print("TOTAL SALARY BY DEPARTMENT:")
print("="*50)
for dept, total in dept_totals:
    print(f"Department: {dept} | Total Salary: {total}")


TOTAL SALARY BY DEPARTMENT:
Department: IT | Total Salary: 170000.0
Department: HR | Total Salary: 85000.0
Department: Finance | Total Salary: 130000.0


In [6]:
# Identify top 3 highest-paid employees
top_3_emp = emp_rdd.takeOrdered(3, key=lambda emp: -emp[3])

# Format output for saving
output_lines = [f"{emp[0]},{emp[1]},{emp[2]},{emp[3]}" for emp in top_3_emp]

# Save to file
output_dir = "top_3_highest_paid"
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

# Force Spark to use exactly 1 partition
output_rdd = sc.parallelize(output_lines).coalesce(1)

# Save the output
output_rdd.saveAsTextFile(output_dir)

print("="*50)
print(f"Top 3 employees saved successfully to directory: '{output_dir}'")
print("="*50)
for line in output_lines:
    print(line)

Top 3 employees saved successfully to directory: 'top_3_highest_paid'
4,Priya,Finance,70000.0
3,Neha,IT,65000.0
7,Rohit,Finance,60000.0
